In [1]:
# %% [markdown]
# # Jupyter Notebook Loading Header
#
# This is a custom loading header for Jupyter Notebooks in Visual Studio Code.
# It includes common imports and settings to get you started quickly.
# %% [markdown]
## Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from google.cloud import storage
import os
import tempfile
import time
from datetime import datetime
import uuid
import joblib
import uuid
from sklearn.metrics import roc_auc_score
from datetime import datetime, timedelta
import gcsfs
import duckdb as dd
import pickle
import joblib
from typing import Union
import io
path = r'C:\Users\Dwaipayan\AppData\Roaming\gcloud\application_default_credentials.json'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = path
client = bigquery.Client(project='prj-prod-dataplatform')
os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"
# %% [markdown]
## Configure Settings
# Set options or configurations as needed
pd.set_option('display.max_columns', None)
pd.set_option("Display.max_rows", 100)

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [2]:
df = pd.read_csv(r"C:\Users\Dwaipayan\D_Drive_Temp\psi_csi_DC\cash_weekly_2023-01-01_to_2026-10-31.csv")
print("The shape of the dataframe is:", df.shape)
df.head()

The shape of the dataframe is: (31395, 16)


,Unnamed: 0,model_name,model_version,trench_category,os_type,feature_name,feat_drift_value,drift_type,run_date,period,data_selection,train_df_count,df_count,remarks,bins_used,unique_bins_used
0,0,apps_score_cash_model,v1,Trench 1,android,prediction,1.754972,psi,2025-09-21,Weekly,prod,219533,213,NaN,10,10
1,1,apps_score_cash_model,v1,Trench 1,android,ml_score,0.579088,csi,2025-09-21,Weekly,prod,219533,213,NaN,10,10
2,2,apps_score_cash_model,v1,Trench 1,android,dl_score,5.068840,csi,2025-09-21,Weekly,prod,219533,213,NaN,10,10
3,3,apps_score_cash_model,v1,Trench 1,android,app_cnt_absence_tag_90d,6.207247,csi,2025-09-21,Weekly,prod,219533,213,NaN,7,8
4,4,apps_score_cash_model,v1,Trench 1,android,app_cnt_rated_for_18plus_ever,1.724968,csi,2025-09-21,Weekly,prod,219533,213,NaN,9,4


In [3]:
df = df[['model_name', 'model_version', 'trench_category',
       'os_type', 'feature_name', 'feat_drift_value', 'drift_type', 'run_date',
       'period', 'data_selection', 'train_df_count', 'df_count', 'remarks',
       'bins_used', 'unique_bins_used']]

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31395 entries, 0 to 31394
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   model_name        31395 non-null  object 
 1   model_version     31395 non-null  object 
 2   trench_category   31395 non-null  object 
 3   os_type           31395 non-null  object 
 4   feature_name      31395 non-null  object 
 5   feat_drift_value  31395 non-null  float64
 6   drift_type        31395 non-null  object 
 7   run_date          31395 non-null  object 
 8   period            31395 non-null  object 
 9   data_selection    31395 non-null  object 
 10  train_df_count    31395 non-null  int64  
 11  df_count          31395 non-null  int64  
 12  remarks           1068 non-null   object 
 13  bins_used         31395 non-null  int64  
 14  unique_bins_used  31395 non-null  int64  
dtypes: float64(1), int64(4), object(10)
memory usage: 3.6+ MB


In [5]:
df['run_date'] = pd.to_datetime(df['run_date']).dt.date

In [6]:

# Convert run_date to datetime
df['run_date'] = pd.to_datetime(df['run_date'])

# Get min and max run_date for each group
summary = (
    df.groupby(['model_name', 'model_version', 'trench_category'])
      .agg(
          min_run_date=('run_date', 'min'),
          max_run_date=('run_date', 'max')
      )
      .reset_index()
)

print(summary)

                model_name model_version trench_category min_run_date  \
0   alpha_stack_cash_model            v1         Overall   2025-09-21   
1   alpha_stack_cash_model            v1        Trench 1   2025-09-21   
2   alpha_stack_cash_model            v1        Trench 2   2025-09-21   
3   alpha_stack_cash_model          v1.1         Overall   2025-12-07   
4   alpha_stack_cash_model          v1.1        Trench 1   2025-12-07   
5   alpha_stack_cash_model          v1.1        Trench 2   2025-12-07   
6    apps_score_cash_model            v1         Overall   2025-09-21   
7    apps_score_cash_model            v1        Trench 1   2025-09-21   
8    apps_score_cash_model            v1        Trench 2   2025-09-21   
9    apps_score_cash_model          v1.1         Overall   2025-12-07   
10   apps_score_cash_model          v1.1        Trench 1   2025-12-07   
11   apps_score_cash_model          v1.1        Trench 2   2025-12-07   
12    beta_demo_cash_model            v1         Ov

In [7]:
# dataset_id = 'dap_ds_poweruser_playground'
# table_id = 'temp_genesysdata'
# # Define the table schema as per your DataFrame columns
# schema = [
#     bigquery.SchemaField("Date", "STRING"),
#     bigquery.SchemaField("DNIS", "STRING"),
#     bigquery.SchemaField("System_Dispositin", "STRING"),
#     bigquery.SchemaField("Wrap_up", "STRING"),
#     bigquery.SchemaField("contact_list_name", "STRING"),
#     bigquery.SchemaField("DPD", "FLOAT64"),
#     bigquery.SchemaField("Bucket", "FLOAT64"),
#     bigquery.SchemaField("phone", "STRING"),
# ]
# # Create the dataset reference
# dataset_ref = client.dataset(dataset_id)
# # Define the table reference
# table_ref = dataset_ref.table(table_id)
# # Configure the job to overwrite the table if it already exists
# job_config = bigquery.LoadJobConfig()
# # Load the DataFrame into BigQuery
# job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
# # Wait for the job to complete
# job.result()
# print(f"Table {table_id} created in dataset {dataset_id}.")

In [8]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.psi_csi_cash_all_models_weekly_v1"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
job.result()  # Wait for the job to complete

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=8cc98bff-9776-4eed-9992-f79e2abb0e49>

# 💻 SIL

In [9]:
dfsil = pd.read_csv(r"C:\Users\Dwaipayan\D_Drive_Temp\psi_csi_DC\sil_weekly_2023-01-01_to_2026-10-31.csv")
print("The shape of the dataframe is:", dfsil.shape)
dfsil.head()

The shape of the dataframe is: (176934, 18)


,Unnamed: 0,model_name,model_version,trench_category,os_type,loan_type,product_type,feature_name,feat_drift_value,drift_type,run_date,period,data_selection,train_df_count,df_count,remarks,bins_used,unique_bins_used
0,0,cic_sil_model,v2,Trench 2,android,SIL-Instore,Appliance,Personal_Loans_granted_contracts_amt_24M,0.0,csi,2025-11-16,Weekly,prod,1940,9,error in csi,5.0,10.0
1,1,cic_sil_model,v2,Trench 2,android,SIL-Instore,Appliance,ScoreRange,0.0,csi,2025-11-16,Weekly,prod,1940,9,error in csi,18.0,18.0
2,2,cic_sil_model,v2,Trench 2,android,SIL-Instore,Appliance,has_ever_been_overdue,0.0,csi,2025-11-16,Weekly,prod,1940,9,error in csi,2.0,1.0
3,3,cic_sil_model,v2,Trench 2,android,SIL-Instore,Appliance,ln_loan_level_user_type,0.0,csi,2025-11-16,Weekly,prod,1940,9,error in csi,1.0,1.0
4,4,cic_sil_model,v2,Trench 2,android,SIL-Instore,Appliance,cnt_nongranted_contracts_3M,0.0,csi,2025-11-16,Weekly,prod,1940,9,error in csi,5.0,2.0


In [10]:
dfsil = dfsil[['model_name', 'model_version', 'trench_category',
       'os_type', 'loan_type', 'product_type', 'feature_name',
       'feat_drift_value', 'drift_type', 'run_date', 'period',
       'data_selection', 'train_df_count', 'df_count', 'remarks', 'bins_used',
       'unique_bins_used']]

In [11]:
dfsil['run_date'] = pd.to_datetime(dfsil['run_date']).dt.date

In [12]:
dfsil.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 176934 entries, 0 to 176933
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   model_name        176934 non-null  object 
 1   model_version     176934 non-null  object 
 2   trench_category   176934 non-null  object 
 3   os_type           176934 non-null  object 
 4   loan_type         176934 non-null  object 
 5   product_type      176934 non-null  object 
 6   feature_name      176934 non-null  object 
 7   feat_drift_value  176934 non-null  float64
 8   drift_type        176934 non-null  object 
 9   run_date          176934 non-null  object 
 10  period            176934 non-null  object 
 11  data_selection    176934 non-null  object 
 12  train_df_count    176934 non-null  int64  
 13  df_count          176934 non-null  int64  
 14  remarks           48562 non-null   object 
 15  bins_used         158441 non-null  float64
 16  unique_bins_used  15

In [13]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.psi_csi_sil_all_models_weekly_v1"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfsil, table_id, job_config=job_config)
job.result()  # Wait for the job to complete

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=91dc9d0e-d3dd-4cfb-8816-6ae0a9483d3d>